<a href="https://colab.research.google.com/github/OdysseusPolymetis/ihrim_demo_24022026/blob/main/pyOdysseus_wtsplit_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# pyOdysseus → Démo web (Gradio)
Entrées : 1 txt pivot + (zip de txt OU multi-fichiers txt)  
Sortie : HTML affiché + fichier .html téléchargeable

In [ ]:
!pip -q install stanza tqdm
!pip -q install faiss-gpu-cu12 || true
!pip install --upgrade gradio

!rm -rf /content/pyOdysseus
!git clone -q https://github.com/OdysseusPolymetis/pyOdysseus.git /content/pyOdysseus

In [ ]:
!pip -q install "wtpsplit[onnx-gpu]"

In [ ]:
import os, glob, pathlib, subprocess, sys

def pip_install_editable(folder):
  p = subprocess.run([sys.executable, "-m", "pip", "install", "-e", "/content/pyOdysseus/bertalign_odysseus"], text=True, capture_output=True)
  print(p.stdout)
  print(p.stderr)
  print("returncode =", p.returncode)
candidates = []
for base in ["/content/pyOdysseus", "/content"]:
    for setup in glob.glob(base + "/**/setup.py", recursive=True):
        candidates.append(str(pathlib.Path(setup).parent))
    for pyproj in glob.glob(base + "/**/pyproject.toml", recursive=True):
        candidates.append(str(pathlib.Path(pyproj).parent))

priority = [p for p in candidates if "bertalign" in p.lower()]
chosen = priority[0] if priority else candidates[0]

pip_install_editable(chosen)

print("OK ✅")

In [ ]:
import sys, importlib

PKG_ROOT = "/content/pyOdysseus/bertalign_odysseus"
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)

for m in list(sys.modules):
    if m == "bertalign" or m.startswith("bertalign."):
        del sys.modules[m]

import bertalign
importlib.reload(bertalign)

print("✅ bertalign importé depuis :", bertalign.__file__)

In [ ]:
from wtpsplit import SaT
sat = SaT("sat-3l-sm")

sat.half().to("cuda")

In [ ]:
SPLIT_THRESHOLD = 0.001

In [ ]:
sample = "Voici une phrase assez longue, avec plusieurs virgules, et peut-être des coupures possibles. Et encore une autre phrase. Puis une troisième."
for th in [0.70, 0.20, 0.10, 0.00001]:
    segs = sat.split(sample, threshold=th)
    print(th, len(segs), segs)

In [ ]:
from lingua import LanguageDetectorBuilder
import stanza

_LANG_DETECTOR = LanguageDetectorBuilder.from_all_languages().build()
_NLP_CACHE = {}

def detect_lang_iso639_1(text: str, default: str = "fr") -> str:
    sample = (text or "").strip()
    if not sample:
        return default
    sample = sample[:5000]
    lang = _LANG_DETECTOR.detect_language_of(sample)
    if lang is None or lang.iso_code_639_1 is None:
        return default
    return lang.iso_code_639_1.name.lower()

def get_stanza_pipeline(lang: str, fallback: str = "fr"):

    lang = (lang or fallback).lower()

    if lang in _NLP_CACHE:
        return _NLP_CACHE[lang]

    def _build(l):
        stanza.download(l, verbose=False)
        return stanza.Pipeline(l, processors="tokenize,pos,lemma", verbose=False)

    try:
        _NLP_CACHE[lang] = _build(lang)
        return _NLP_CACHE[lang]
    except Exception as e:

        if fallback not in _NLP_CACHE:
            _NLP_CACHE[fallback] = _build(fallback)
        return _NLP_CACHE[fallback]

def lemmatize_phrase_auto(phrase: str, lang: str):
    nlp = get_stanza_pipeline(lang)
    doc = nlp(phrase)
    tokens = []
    for sent in doc.sentences:
        for token in sent.tokens:
            for w in token.words:
                tokens.append((w.text, w.lemma))
    return tokens

In [ ]:
import os, re, glob, zipfile, tempfile, pathlib
from collections import Counter
from tqdm import tqdm

import gradio as gr

from bertalign import Bertalign
print("Import OK", Bertalign)

In [ ]:
def split_chants(text: str):

    pattern = re.compile(r'^(Chant\s*\d+)', re.IGNORECASE)
    chants, current = [], []
    for line in text.splitlines():
        s = line.strip()
        if pattern.match(s):
            if current:
                chants.append("\n".join(current))
                current = []
        current.append(s)
    if current:
        chants.append("\n".join(current))
    return chants if chants else [text]

def preprocess_chants(text_ids, src_folder, chants_folder):

    os.makedirs(chants_folder, exist_ok=True)
    for text_id in tqdm(text_ids, desc="Découpage en chants", unit="texte"):
        path = os.path.join(src_folder, f"{text_id}.txt")
        if not os.path.exists(path):
            print(f"[WARN] Introuvable: {path}")
            continue
        content = pathlib.Path(path).read_text(encoding="utf-8", errors="ignore")
        chants = split_chants(content)
        for i, chant in enumerate(chants, start=1):
            out = os.path.join(chants_folder, f"{text_id}_Chant{i}.txt")
            pathlib.Path(out).write_text(chant, encoding="utf-8")


In [ ]:
import urllib.request

stopwords_url = "https://raw.githubusercontent.com/OdysseusPolymetis/colabs_for_nlp/refs/heads/main/stopwords_fr.txt"
stopwords_path = "/content/stopwords_fr.txt"
urllib.request.urlretrieve(stopwords_url, stopwords_path)

STOPWORDS = pathlib.Path(stopwords_path).read_text(encoding="utf-8", errors="ignore").splitlines()

In [ ]:
def align_pairwise(src_sents, tgt_sents):
    src_text = "\n".join(src_sents)
    tgt_text = "\n".join(tgt_sents)
    aligner = Bertalign(src_text, tgt_text, is_split=True)
    return aligner.align_sents()


from typing import Dict, List, Tuple




from typing import Dict, List, Tuple

def build_cells_alignment(
    pivot_id: str,
    text_ids: List[str],
    chant_number: int,
    all_sents: Dict[str, List[str]],
    max_align: int = 5,
    top_k: int = 3,
    win: int = 5,
    skip: float = -0.1,
    margin: bool = True,
    len_penalty: bool = True,
) -> Tuple[List[str], Dict[str, List[object]]]:
    """
    Retourne:
    - pivot_sents : liste des segments pivot
    - cells_by_tid : dict {tid: cells} pour tid != pivot_id
        cells = liste longueur len(pivot_sents) (None / 0 / {"rowspan","idxs"})
    """
    if pivot_id not in all_sents or not all_sents[pivot_id]:
        raise ValueError(f"Pivot manquant/vide dans all_sents: {pivot_id} (chant {chant_number}).")

    pivot_sents = all_sents[pivot_id]
    nb_pivot = len(pivot_sents)

    other_ids = [tid for tid in text_ids if tid != pivot_id and tid in all_sents and all_sents[tid]]
    cells_by_tid: Dict[str, List[object]] = {}

    for tid in tqdm(other_ids, desc=f"Align pivot→others (Chant {chant_number})", unit="texte"):
        tgt_sents = all_sents[tid]

        aligner = Bertalign(
            pivot_sents,
            tgt_sents,
            max_align=max_align,
            top_k=top_k,
            win=win,
            skip=skip,
            margin=margin,
            len_penalty=len_penalty,
            is_split=True,
        ).align_sents()

        beads = getattr(aligner, "result", [])
        cells_by_tid[tid] = beads_to_cells(beads, nb_src=nb_pivot)

    return pivot_sents, cells_by_tid



In [ ]:
def new_get_freq_class(lemma, local_counter, lemma_authors, n_authors):
    if lemma in STOPWORDS:
        return "freq0"
    if local_counter.get(lemma, 0) == 1:
        return "freq1"
    authors_count = len(lemma_authors.get(lemma, []))
    if authors_count < n_authors / 4:
        return "freq2"
    if authors_count >= 0.7 * n_authors:
        if authors_count >= n_authors - 2:
            return "freq5"
        else:
            return "freq4"
    return "freq3"

def compute_local_lemma_freq_and_authors(row, all_sents, all_tokens):
    freq = Counter()
    lemma_authors = {}
    for text_id, indices in row.items():
        if isinstance(indices, int):
            indices = [indices]
        if not indices:
            continue
        for idx in indices:
            if text_id not in all_tokens or idx >= len(all_tokens[text_id]):
                continue
            for forme, lemma in all_tokens[text_id][idx]:
                freq[lemma] += 1
                lemma_authors.setdefault(lemma, set()).add(text_id)
    return freq, lemma_authors

def render_segment(text_id, idx, all_sents, local_counter, n_authors, lemma_authors, all_tokens):
    if text_id not in all_tokens or idx >= len(all_tokens[text_id]):
        return f"<span style='color:red;'>(Segment {idx} introuvable pour {text_id})</span>"
    tokens = all_tokens[text_id][idx]
    out = []
    for forme, lemma in tokens:
        cls = new_get_freq_class(lemma, local_counter, lemma_authors, n_authors)
        tooltip = ""
        if cls != "freq0" and local_counter.get(lemma, 0) <= 1 and lemma in lemma_authors:
            tooltip = " title='Found in: " + ", ".join(sorted(lemma_authors[lemma])) + "'"
        out.append(f'<mark class="{cls}"{tooltip}>{forme}</mark>')
    return " ".join(out)

CSS_MINI = """
<style>
body { font-family: system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif; margin: 16px; }
h1,h2 { margin: 0.6rem 0; }
.small { color:#666; font-size: 0.9rem; }
table { width:100%; border-collapse: collapse; table-layout: fixed; }
th, td { border: 1px solid #ddd; vertical-align: top; padding: 8px; word-wrap: break-word; }
th { position: sticky; top: 0; background: #fafafa; z-index: 1; }
.blockno { width: 70px; text-align: right; color:#555; }
mark { padding: 0 2px; border-radius: 3px; }
.freq0 { background: transparent; color: #111; }
.freq1 { background: #ffd6d6; }     /* rare local */
.freq2 { background: #ff9a9a; }     /* "plag" */
.freq3 { background: #f3f3f3; }     /* neutre */
.freq4 { background: #d6f0ff; }     /* fréquent */
.freq5 { background: #c7ffc7; }     /* très fréquent */
</style>
"""

def generate_html_cells_only(
    pivot_id,
    text_ids,
    chant_number,
    pivot_sents,
    cells_by_tid,
    all_sents,
    all_tokens=None,
    highlight_pivot=True,
):
    """
    Génère un HTML avec :
    - pivot en 1ère colonne (à gauche)
    - colonnes cibles avec fusion (rowspan) selon cells_by_tid
    - dans chaque cellule cible : préfixe textuel des indices source (pivot) entre crochets
      ex: [12] ou [12-13] devant chaque segment cible rendu.
    """
    n_authors = len(text_ids)
    ordered_ids = [pivot_id] + [tid for tid in text_ids if tid != pivot_id]
    other_ids = [tid for tid in ordered_ids if tid != pivot_id]

    # Header
    cols = "".join([f"<th>{tid}</th>" for tid in ordered_ids])
    rows_html = []

    for i in range(len(pivot_sents)):
        # Construire une "row" (pivot i) pour les stats de fréquence
        row_like = {pivot_id: i}
        for tid in other_ids:
            col = cells_by_tid.get(tid, [None] * len(pivot_sents))
            cell = col[i] if i < len(col) else None

            if isinstance(cell, dict):
                row_like[tid] = cell.get("idxs", [])
            else:
                row_like[tid] = []

        local_counter, local_lemma_authors = compute_local_lemma_freq_and_authors(
            row_like, all_sents, all_tokens
        )

        tds = []

        # --- Pivot cell
        if highlight_pivot:
            pivot_html = render_segment(
                pivot_id,
                i,
                all_sents,
                local_counter,
                n_authors,
                local_lemma_authors,
                all_tokens,
            )
            tds.append(f"<td>{pivot_html}</td>")
        else:
            tds.append(f"<td>{pivot_sents[i]}</td>")

        # --- Target cells
        for tid in other_ids:
            col = cells_by_tid.get(tid, [None] * len(pivot_sents))
            cell = col[i] if i < len(col) else None

            # 0 = couvert par un rowspan démarré plus haut
            if cell == 0:
                continue

            if cell is None:
                tds.append("<td>∅</td>")
                continue

            idxs = [int(x) for x in cell.get("idxs", [])]
            rowspan = int(cell.get("rowspan", 1))

            # Tag d'indices source (1-based pour l'affichage)
            if rowspan > 1:
                src_tag = f"[{i+1}-{i+rowspan}]"
            else:
                src_tag = f"[{i+1}]"

            if not idxs:
                content = "∅"
            else:
                segs = []
                for j in idxs:
                    # Sécurité bornes
                    if tid not in all_sents or j < 0 or j >= len(all_sents[tid]):
                        continue

                    seg_html = render_segment(
                        tid,
                        j,
                        all_sents,
                        local_counter,
                        n_authors,
                        local_lemma_authors,
                        all_tokens,
                    )
                    segs.append(f"<span class='srcidx'>{src_tag}</span> {seg_html}")

                content = "<br/>".join(segs) if segs else "∅"

            rs = f' rowspan="{rowspan}"' if rowspan > 1 else ""
            tds.append(f"<td{rs}>{content}</td>")

        rows_html.append(
            "<tr>"
            + f"<td class='blockno'><b>{i+1}</b></td>"
            + "".join(tds)
            + "</tr>"
        )

    html = f"""
    <html>
      <head>
        <meta charset="utf-8">
        <title>Alignement Chant {chant_number}</title>
        {CSS_MINI}
        <style>
          .srcidx {{
            color:#666;
            font-size:0.85em;
            font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
            margin-right: 6px;
            white-space: nowrap;
          }}
        </style>
      </head>
      <body>
        <h1>Alignement — Chant {chant_number}</h1>
        <div class="small">Pivot: <b>{pivot_id}</b> • Textes: {", ".join(ordered_ids)}</div>

        <table>
          <thead>
            <tr><th class="blockno">#</th>{cols}</tr>
          </thead>
          <tbody>
            {''.join(rows_html)}
          </tbody>
        </table>
      </body>
    </html>
    """
    return html


In [ ]:
def beads_to_cells(beads, nb_src: int):

    cells = [None] * nb_src
    pending = []
    last_anchor = None

    def attach(anchor, tgts):
        if anchor is None:
            return
        if cells[anchor] in (None, 0):
            cells[anchor] = {"rowspan": 1, "idxs": []}
        seen = set(cells[anchor]["idxs"])
        for t in tgts:
            t = int(t)
            if t not in seen:
                cells[anchor]["idxs"].append(t)
                seen.add(t)

    for sr, tr in beads:
        sr = [int(x) for x in sr]
        tr = [int(x) for x in tr]

        if not sr and tr:
            pending.extend(tr)
            continue

        if sr:
            top = sr[0]
            bot = sr[-1]
            rowspan = max(1, bot - top + 1)

            if pending:
                tr = pending + tr
                pending = []

            if cells[top] in (None, 0):
                cells[top] = {"rowspan": rowspan, "idxs": []}
            else:
                cells[top]["rowspan"] = max(cells[top]["rowspan"], rowspan)

            attach(top, tr)

            for r in range(top + 1, min(nb_src, top + rowspan)):
                cells[r] = 0

            last_anchor = top

    if pending:
        anchor = last_anchor if last_anchor is not None else (nb_src - 1 if nb_src else None)
        attach(anchor, pending)

    return cells


In [ ]:
import re

_PUNCT_ONLY = re.compile(r"^[\s\W_]+$")

def split_units_clean(text: str):
    segs = sat.split(text, threshold=SPLIT_THRESHOLD)
    segs = [s.strip() for s in segs]
    segs = [s for s in segs if s]

    merged = []
    for s in segs:

        if merged and (len(s) <= 3 or _PUNCT_ONLY.match(s)):
            merged[-1] = (merged[-1] + " " + s).strip()
        else:
            merged.append(s)
    return merged

In [ ]:
def extract_zip_to_dir(zip_path: str, dst_dir: str):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(dst_dir)

def infer_text_id_from_filename(path: str) -> str:

    return pathlib.Path(path).stem.strip()

def run_app(source_file, corpus_files):

    if source_file is None:
        raise gr.Error("Il faut fournir le fichier source .txt (pivot).")
    if not corpus_files:
        raise gr.Error("Il faut fournir au moins un fichier cible .txt.")

    with tempfile.TemporaryDirectory() as tmp:
        tmp = pathlib.Path(tmp)
        src_dir = tmp / "source"
        chants_dir = tmp / "chants"
        src_dir.mkdir(parents=True, exist_ok=True)
        chants_dir.mkdir(parents=True, exist_ok=True)

        pivot_id = infer_text_id_from_filename(source_file.name)
        pivot_path = src_dir / f"{pivot_id}.txt"
        pivot_path.write_bytes(pathlib.Path(source_file.name).read_bytes())


        if corpus_files:
            for f in corpus_files:
                dst = src_dir / pathlib.Path(f.name).name
                dst.write_bytes(pathlib.Path(f.name).read_bytes())
        else:
            raise gr.Error("Il faut fournir au moins un fichier cible txt.")

        txt_files = sorted([p for p in src_dir.rglob("*.txt")])
        text_ids = []
        for p in txt_files:
            tid = infer_text_id_from_filename(str(p))
            if tid not in text_ids:
                text_ids.append(tid)

        if pivot_id not in text_ids:
            text_ids.insert(0, pivot_id)

        preprocess_chants(text_ids, str(src_dir), str(chants_dir))
        print("preprocessing des chants fait")

        pivot_chants = sorted(glob.glob(str(chants_dir / f"{pivot_id}_Chant*.txt")))
        if not pivot_chants:
            raise gr.Error("Aucun chant détecté pour le pivot. Vérifie le contenu / format du .txt.")

        chant_numbers = sorted([
            int(re.search(r"Chant(\d+)", pathlib.Path(f).name).group(1))
            for f in pivot_chants
        ])

        html_parts = []
        for chant_number in chant_numbers:
            all_sents = {}
            all_langs = {}

            for tid in text_ids:
              p = chants_dir / f"{tid}_Chant{chant_number}.txt"
              if not p.exists():
                continue
              content = p.read_text(encoding="utf-8", errors="ignore")

              all_langs[tid] = detect_lang_iso639_1(content, default="fr")
              all_sents[tid] = split_units_clean(content)


            all_tokens = {}
            for tid, segs in all_sents.items():
              lang = all_langs.get(tid, "fr")
              all_tokens[tid] = [lemmatize_phrase_auto(s, lang) for s in segs]

            pivot_sents, cells_by_tid = build_cells_alignment(
              pivot_id=pivot_id,
              text_ids=text_ids,
              chant_number=chant_number,
              all_sents=all_sents,
            )

            html_parts.append(
              generate_html_cells_only(
              pivot_id, text_ids, chant_number,
              pivot_sents, cells_by_tid,
              all_sents, all_tokens
              )
            )


        final_html = "\n<hr/>\n".join(html_parts)

        out_path = pathlib.Path("/content/resultat_alignement.html")
        out_path.write_text(final_html, encoding="utf-8")
    return final_html, str(out_path)


with gr.Blocks(title="pyOdysseus — Alignement → HTML") as demo:
    gr.Markdown("# Alignement → HTML (démo)\nPivot (.txt) + corpus (.zip de .txt ou multi-fichiers) → rendu HTML.")

    with gr.Row():
        source = gr.File(label="Entrée 1 — Pivot / Source (.txt)", file_types=[".txt"])
        corpus_files = gr.Files(label="Entrée 2 — Textes cibles (.txt, multi-fichiers)", file_types=[".txt"])

    btn = gr.Button("Générer")
    html_out = gr.HTML(label="Aperçu HTML")
    file_out = gr.File(label="Télécharger le .html")

    btn.click(run_app, inputs=[source, corpus_files], outputs=[html_out, file_out])

app = demo.queue(max_size=20).launch(
    share=True,
    debug=True,
    prevent_thread_lock=True,
)
print("Gradio lancé ✅")
